In [58]:
from common.aws_config import aws_env_setup
from common.bedrock_config import get_anthropic_client
import json 

aws_env_setup()
client, model = get_anthropic_client()

In [59]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        #"temperature": temperature
        "extra_body": {"temperature": temperature}
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

Model Grader 

In [60]:
def grade_by_model(test_case, output):
    # create evaluation prompt
    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI generated solution.
<task>
Task: {test_case['task']}
</task>

Solution to evaluate:

<solution>
Solution: {output}
</solution>

Criteria you shoul use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Provide evaluation as a JSON object with exactly these keys:
- "score": a number between 1 and 10
- "reasoning": a concise explanation of your assessment
- "strengths": an array of 1-3 key strengths
- "weaknesses": an array of 1-3 key areas for improvement
"""

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    model_grade = json.loads(eval_text)

    return {
        "score": model_grade.get("score", model_grade.get("Score")),
        "reasoning": model_grade.get("reasoning", model_grade.get("Reasoning", "")),
        "strengths": model_grade.get("strengths", model_grade.get("Strengths", [])),
        "weaknesses": model_grade.get("weaknesses", model_grade.get("Weaknesses", model_grade.get("Weakness", []))),
    }

In [61]:
# run_prompt funtion 
def run_prompt(test_case):
    "Merges Prompt + test case input - Returns result"
    prompt = f"""
             Solve the following task:
             {test_case["task"]}
            - Respond only with Python , JSON or a plain Regex 
            - Do not addd any comments or commentary or explanation 
             """ 
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, temperature=0.7, stop_sequences=["```"])
    return output
 

In [62]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
        "solution_criteria": "Key criteria for evaluation the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [63]:
import json 
import ast
import re 

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case ["format"]
    if format == 'json' :
        return validate_json(response)
    elif format == 'python':
        return validate_python(response)
    else:
        return validate_regex(response)
    
        

In [64]:
def run_test_case(test_case):
    output = run_prompt(test_case)

    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    # 
    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score)/2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
        "strengths": model_grade.get("strengths", []),
        "weaknesses": model_grade.get("weaknesses", []),
    }

In [65]:
from statistics import mean

def run_eval(dataset):
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [66]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    dataset = json.dump(dataset,f,indent=2)
    #print(dataset)
#results = run_eval(dataset)


Code based grader + model based grader average 


In [67]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
    #print(dataset)
results = run_eval(dataset)

Average score: 8.833333333333334


In [ ]:
print(json.dumps(results , indent=2))

[
  {
    "output": "\nimport re\nimport json\nfrom datetime import datetime\n\ndef parse_cloudwatch_log(log_entry):\n    pattern = r'(\\d{4}-\\d{2}-\\d{2}T\\d{2}:\\d{2}:\\d{2}\\.\\d+Z)\\s+(\\w+)\\s+(.*)'\n    match = re.match(pattern, log_entry)\n    \n    if match:\n        return {\n            \"timestamp\": match.group(1),\n            \"log_level\": match.group(2),\n            \"message\": match.group(3)\n        }\n    return None\n\n# Example usage\nlog_entry = \"2024-01-15T10:30:45.123Z ERROR Failed to connect to database\"\nresult = parse_cloudwatch_log(log_entry)\nprint(json.dumps(result, indent=2))\n",
    "test_case": {
      "task": "Parse an AWS CloudWatch log entry and extract the timestamp, log level, and message components",
      "format": "regex",
      "solution_criteria": "The regex should correctly capture timestamp in ISO 8601 format, log level (INFO, ERROR, WARN, DEBUG), and the remaining message text from a typical CloudWatch log format like '2024-01-15T10:30

: 